"""
Project 2: Supervised Learning - Fraud Detection Pipeline
DecodeLabs Industrial Training Kit

Dataset required: creditcard.csv (Credit Card Fraud Detection - ULB)
Download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
Place the file in the same folder as this script (or update DATA_PATH below).

Implements the "Zero-Leakage Protocol":
    1. Split FIRST, resample LATER (no leakage into the test set)
    2. Use imblearn.pipeline.Pipeline (not sklearn's) so SMOTE is applied
       only inside each training fold, never on validation/test data
    3. Train both Logistic Regression (with scaling) and Random Forest
       (no scaling needed - tree splits are scale-invariant)
    4. Tune hyperparameters (including SMOTE's k_neighbors) with GridSearchCV
    5. Evaluate using Precision, Recall, ROC-AUC and Confusion Matrix
       (accuracy is intentionally NOT used as the deciding metric)
"""


"""
Project 2: Supervised Learning - Fraud Detection Pipeline
DecodeLabs Industrial Training Kit

Dataset required: creditcard.csv (Credit Card Fraud Detection - ULB)
Download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
Place the file in the same folder as this script (or update DATA_PATH below).

Implements the "Zero-Leakage Protocol":
    1. Split FIRST, resample LATER (no leakage into the test set)
    2. Use imblearn.pipeline.Pipeline (not sklearn's) so SMOTE is applied
       only inside each training fold, never on validation/test data
    3. Train both Logistic Regression (with scaling) and Random Forest
       (no scaling needed - tree splits are scale-invariant)
    4. Tune hyperparameters (including SMOTE's k_neighbors) with GridSearchCV
    5. Evaluate using Precision, Recall, ROC-AUC and Confusion Matrix
       (accuracy is intentionally NOT used as the deciding metric)
"""


In [4]:
import pandas as pd

# Original dataset load karo
df = pd.read_csv('creditcard.csv')

print("Original shape:", df.shape)
print("Original fraud count:", df['Class'].sum())

# Stratified sampling - Class ratio maintain karega
sample_size = 50000
fraction = sample_size / len(df)

# Fraud aur legit ko alag-alag sample karo, phir combine
df_fraud = df[df['Class'] == 1].sample(frac=fraction, random_state=42)
df_legit = df[df['Class'] == 0].sample(frac=fraction, random_state=42)

df_sampled = pd.concat([df_fraud, df_legit])

# Shuffle karo taaki rows mixed order me ho
df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nNew shape:", df_sampled.shape)
print("New fraud count:", df_sampled['Class'].sum())
print("New fraud %:", round(df_sampled['Class'].mean() * 100, 4))

# Save kar do naya CSV
df_sampled.to_csv('creditcard_50k.csv', index=False)
print("\nSaved as creditcard_50k.csv")

Original shape: (284807, 31)
Original fraud count: 492

New shape: (50000, 31)
New fraud count: 86
New fraud %: 0.172

Saved as creditcard_50k.csv


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

from imblearn.pipeline import Pipeline  # <-- imblearn's Pipeline, NOT sklearn's
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
DATA_PATH = "creditcard_50k.csv"  # update path if needed


# ---------------------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------------------
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
    print("Class distribution:")
    print(df["Class"].value_counts(normalize=True) * 100)
    return df


# ---------------------------------------------------------------------------
# 2. TRAIN / TEST SPLIT  (done BEFORE any resampling -> avoids leakage)
# ---------------------------------------------------------------------------
def split_data(df: pd.DataFrame):
    X = df.drop(columns=["Class"])
    y = df["Class"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        stratify=y,           # keep the same fraud ratio in both sets
        random_state=RANDOM_STATE,
    )
    print(f"\nTrain size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")
    print("Train fraud rate:", y_train.mean() * 100, "%")
    print("Test fraud rate :", y_test.mean() * 100, "%  (kept realistically imbalanced)")
    return X_train, X_test, y_train, y_test


# ---------------------------------------------------------------------------
# 3. BUILD PIPELINES
# ---------------------------------------------------------------------------
def build_logreg_pipeline() -> Pipeline:
    """Logistic Regression needs scaling -> Scaler -> SMOTE -> Classifier."""
    return Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])


def build_rf_pipeline() -> Pipeline:
    """Random Forest is scale-invariant -> SMOTE -> Classifier (no scaler)."""
    return Pipeline(steps=[
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    ])


# ---------------------------------------------------------------------------
# 4. GRID SEARCH (SMOTE is re-applied safely inside every CV fold)
# ---------------------------------------------------------------------------
def tune_logreg(X_train, y_train):
    pipeline = build_logreg_pipeline()
    param_grid = {
        "smote__k_neighbors": [3, 5, 7],
        "classifier__C": [0.01, 0.1, 1.0],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(
        pipeline, param_grid,
        scoring="roc_auc",   # optimize for ROC-AUC, not accuracy
        cv=cv, n_jobs=-1, verbose=1,
    )
    grid.fit(X_train, y_train)
    print("\n[Logistic Regression] Best params:", grid.best_params_)
    print("[Logistic Regression] Best CV ROC-AUC:", grid.best_score_)
    return grid.best_estimator_


def tune_rf(X_train, y_train):
    pipeline = build_rf_pipeline()
    param_grid = {
        "smote__k_neighbors": [3, 5],
        "classifier__max_depth": [10, 20, None],
        "classifier__n_estimators": [100, 200],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(
        pipeline, param_grid,
        scoring="roc_auc",
        cv=cv, n_jobs=-1, verbose=1,
    )
    grid.fit(X_train, y_train)
    print("\n[Random Forest] Best params:", grid.best_params_)
    print("[Random Forest] Best CV ROC-AUC:", grid.best_score_)
    return grid.best_estimator_


# ---------------------------------------------------------------------------
# 5. EVALUATION  (Precision, Recall, ROC-AUC, Confusion Matrix - NOT accuracy)
# ---------------------------------------------------------------------------
def evaluate_model(model, X_test, y_test, model_name: str):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f"\n===== {model_name} — Test Set Evaluation =====")
    print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

    roc_auc = roc_auc_score(y_test, y_proba)
    avg_precision = average_precision_score(y_test, y_proba)
    print(f"ROC-AUC       : {roc_auc:.4f}")
    print(f"Average Prec. : {avg_precision:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:\n", cm)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Fraud"])
    disp.plot(cmap="Blues")
    plt.title(f"{model_name} - Confusion Matrix")
    plt.savefig(f"{model_name.lower().replace(' ', '_')}_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.close()

    return {"roc_auc": roc_auc, "avg_precision": avg_precision, "y_proba": y_proba}


def plot_roc_comparison(results: dict, y_test):
    plt.figure(figsize=(7, 6))
    for name, res in results.items():
        fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
        plt.plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Random guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve Comparison")
    plt.legend()
    plt.savefig("roc_curve_comparison.png", dpi=150, bbox_inches="tight")
    plt.close()


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------
def main():
    df = load_data(DATA_PATH)
    X_train, X_test, y_train, y_test = split_data(df)

    print("\n" + "=" * 60)
    print("Tuning Logistic Regression (Scaler -> SMOTE -> Classifier)")
    print("=" * 60)
    best_logreg = tune_logreg(X_train, y_train)

    print("\n" + "=" * 60)
    print("Tuning Random Forest (SMOTE -> Classifier)")
    print("=" * 60)
    best_rf = tune_rf(X_train, y_train)

    results = {}
    results["Logistic Regression"] = evaluate_model(best_logreg, X_test, y_test, "Logistic Regression")
    results["Random Forest"] = evaluate_model(best_rf, X_test, y_test, "Random Forest")

    plot_roc_comparison(results, y_test)

    print("\nAll done. Confusion matrix images and ROC comparison saved to disk.")


if __name__ == "__main__":
    main()

Loaded dataset: 50000 rows, 31 columns
Class distribution:
Class
0    99.828
1     0.172
Name: proportion, dtype: float64

Train size: 40000  |  Test size: 10000
Train fraud rate: 0.1725 %
Test fraud rate : 0.16999999999999998 %  (kept realistically imbalanced)

Tuning Logistic Regression (Scaler -> SMOTE -> Classifier)
Fitting 5 folds for each of 9 candidates, totalling 45 fits

[Logistic Regression] Best params: {'classifier__C': 0.01, 'smote__k_neighbors': 5}
[Logistic Regression] Best CV ROC-AUC: 0.9817253155335596

Tuning Random Forest (SMOTE -> Classifier)
Fitting 5 folds for each of 12 candidates, totalling 60 fits
